<a href="https://colab.research.google.com/github/Miranda09041966/RainFall/blob/main/C%C3%B3pia_de_DadaBaseSQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
import pandas as pd

# 1. Criando a conexão com um banco de dados SQLite na memória
conexao = sqlite3.connect(':memory:')
cursor = conexao.cursor()

# 2. Criando as estruturas das tabelas (DDL)
cursor.executescript('''
CREATE TABLE cliente (
    id_cliente INT PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    cidade VARCHAR(50)
);

CREATE TABLE produto (
    id_produto INT PRIMARY KEY,
    nome VARCHAR(100) NOT NULL,
    categoria VARCHAR(50),
    preco_atual DECIMAL(10, 2) NOT NULL
);

CREATE TABLE pedido (
    id_pedido INT PRIMARY KEY,
    id_cliente INT NOT NULL,
    data_pedido DATE NOT NULL,
    status VARCHAR(50) NOT NULL,
    FOREIGN KEY (id_cliente) REFERENCES cliente(id_cliente)
);

CREATE TABLE item_pedido (
    id_pedido INT NOT NULL,
    id_produto INT NOT NULL,
    quantidade INT NOT NULL,
    preco_unitario DECIMAL(10, 2) NOT NULL,
    PRIMARY KEY (id_pedido, id_produto),
    FOREIGN KEY (id_pedido) REFERENCES pedido(id_pedido),
    FOREIGN KEY (id_produto) REFERENCES produto(id_produto)
);
''')

# 3. Gerando e Inserindo um Dataset mais robusto para análise (CORRIGIDO)
clientes_dados = [
    (1, 'Fulano da Silva', 'São Paulo'),
    (2, 'Maria Souza', 'Rio de Janeiro'),
    (3, 'Carlos Oliveira', 'Belo Horizonte'),
    (4, 'Ana Costa', 'São Paulo'),
    (5, 'Lucas Lima', 'Curitiba')
]
cursor.executemany('INSERT INTO cliente VALUES (?, ?, ?)', clientes_dados)

produtos_dados = [
    (101, 'Mouse Sem Fio', 'Periféricos', 89.90),
    (102, 'Teclado Mecânico', 'Periféricos', 250.00),
    (103, 'Monitor 24"', 'Monitores', 899.00),
    (104, 'Cadeira Gamer', 'Móveis', 1200.00),
    (105, 'Fone de Ouvido Bluetooth', 'Áudio', 150.00)
]
cursor.executemany('INSERT INTO produto VALUES (?, ?, ?, ?)', produtos_dados)

pedidos_dados = [
    (5001, 1, '2026-07-01', 'Finalizado'),
    (5002, 2, '2026-07-02', 'Finalizado'),
    (5003, 3, '2026-07-03', 'Em Processamento'),
    (5004, 4, '2026-07-04', 'Finalizado'),
    (5005, 1, '2026-07-05', 'Finalizado'),
    (5006, 5, '2026-07-05', 'Cancelado')
]
# Agora enviando exatamente as 4 colunas que a tabela espera:
cursor.executemany('INSERT INTO pedido VALUES (?, ?, ?, ?)', pedidos_dados)

# Itens dos pedidos (id_pedido, id_produto, quantidade, preco_unitario)
itens_dados = [
    (5001, 101, 2, 89.90),
    (5001, 102, 1, 250.00),
    (5002, 102, 1, 250.00),
    (5003, 103, 1, 899.00),
    (5004, 104, 1, 1200.00),
    (5004, 105, 2, 150.00),
    (5005, 105, 1, 150.00),
    (5006, 101, 1, 89.90)
]
cursor.executemany('INSERT INTO item_pedido VALUES (?, ?, ?, ?)', itens_dados)
conexao.commit()

# ==============================================================================
# 4. EXTRAINDO OS DADOS COM PYTHON + PANDAS (O seu Dataset de Análise)
# ==============================================================================

# Query SQL para juntar tudo em um único "grupão" de dados analíticos
query_analitica = '''
SELECT
    p.id_pedido,
    p.data_pedido,
    p.status,
    c.nome AS nome_cliente,
    c.cidade AS cidade_cliente,
    prod.nome AS nome_produto,
    prod.categoria AS categoria_produto,
    i.quantidade,
    i.preco_unitario,
    (i.quantidade * i.preco_unitario) AS subtotal
FROM pedido p
JOIN cliente c ON p.id_cliente = c.id_cliente
JOIN item_pedido i ON p.id_pedido = i.id_pedido
JOIN produto prod ON i.id_produto = prod.id_produto;
'''

# O Pandas lê o banco de dados e transforma em um DataFrame
df_vendas = pd.read_sql_query(query_analitica, conexao)

# Mostra as primeiras linhas do dataset no Python
print("Visualizando o DataFrame resultante no Python:")
print(df_vendas.head())

# Fechando a conexão após o uso
conexao.close()

Visualizando o DataFrame resultante no Python:
   id_pedido data_pedido            status     nome_cliente  cidade_cliente  \
0       5001  2026-07-01        Finalizado  Fulano da Silva       São Paulo   
1       5001  2026-07-01        Finalizado  Fulano da Silva       São Paulo   
2       5002  2026-07-02        Finalizado      Maria Souza  Rio de Janeiro   
3       5003  2026-07-03  Em Processamento  Carlos Oliveira  Belo Horizonte   
4       5004  2026-07-04        Finalizado        Ana Costa       São Paulo   

       nome_produto categoria_produto  quantidade  preco_unitario  subtotal  
0     Mouse Sem Fio       Periféricos           2            89.9     179.8  
1  Teclado Mecânico       Periféricos           1           250.0     250.0  
2  Teclado Mecânico       Periféricos           1           250.0     250.0  
3       Monitor 24"         Monitores           1           899.0     899.0  
4     Cadeira Gamer            Móveis           1          1200.0    1200.0  
